# Sprint 4 - PlantDoc Fine-Tuning & the Real-World Gap

**Goal (see AGENT.md / final_brief_and_plan.md):** the whole thesis. A model trained on clean lab
photos (PlantVillage) underperforms on real-world field photos (PlantDoc). This sprint:

1. **Measures the gap** - score the Sprint 1 baseline on PlantDoc field photos.
2. **Stage 2 fine-tuning** (variant 3 `pv_plus_plantdoc`): warm-start from the baseline, unfreeze the
   last backbone blocks, fine-tune at a low LR on PlantDoc (classes mapped to PlantVillage labels).
3. **Both** (variant 4 `both`): same fine-tune **plus** augmentation.
4. Checks the fine-tuned model still works on PlantVillage (no catastrophic forgetting).

PlantDoc classes are mapped to PlantVillage classes via `class_map.json` (`PLANTDOC_TO_PLANTVILLAGE`
in `scripts/organize_datasets.py`), so the same 38-class head is used everywhere - that is what makes
the cross-dataset numbers comparable.

In [ ]:
import platform
import subprocess
import sys

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True, timeout=30)
    print(gpu.stdout.strip().splitlines()[0] if gpu.stdout.strip() else gpu.stderr.strip() or "No GPU detected (CPU only)")
except Exception as exc:
    print("GPU check skipped:", exc)

## Step 1 - Mount Drive + clone repo

Requires the Sprint 0 archives (`plantvillage_raw.zip` + `plantdoc_raw.zip` + manifests) on Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

REPO_URL = "https://github.com/io-PEAK/folium.git"   # change if you forked
REPO_DIR = Path("/content/folium")
DATA_DIR = Path("/content/drive/MyDrive/folium/data")    # durable archives (from Sprint 0)
LOCAL_RAW_DIR = Path("/content/folium_raw")             # per-session raw
LOCAL_DATA_DIR = Path("/content/folium_data")            # per-session organized splits
CHECKPOINT_DIR = Path("/content/drive/MyDrive/folium/checkpoints")
RESULTS_DIR = Path("/content/drive/MyDrive/folium/results")

if not (REPO_DIR / "ml").exists():
    %cd /content
    !git clone --depth 1 {REPO_URL}
else:
    !git -C {REPO_DIR} pull --ff-only -q

for d in (DATA_DIR, LOCAL_RAW_DIR, LOCAL_DATA_DIR, CHECKPOINT_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)
print("DATA_DIR (archives):", DATA_DIR)
print("LOCAL_DATA_DIR:", LOCAL_DATA_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)
print("RESULTS_DIR:", RESULTS_DIR)

## Step 2 - Install dependencies

In [ ]:
%pip install -q --upgrade pip
%pip install -q torch torchvision albumentations matplotlib pandas tqdm opencv-python-headless scikit-learn

## Step 3 - Hydrate raw from Drive, then organize splits locally

Same as Sprint 0/1/3: unzip both archives from Drive into local raw, then build
`train/val/test` folders with `scripts/organize_datasets.py` (seed 42, deterministic) plus
`class_map.json`. Idempotent - re-running is harmless.

In [ ]:
import sys

sys.path.insert(0, str(REPO_DIR))
from scripts.download_datasets import PLANTVILLAGE_EXPECTED, PLANTDOC_EXPECTED, hydrate_dataset

for name, expected in (("plantvillage", PLANTVILLAGE_EXPECTED), ("plantdoc", PLANTDOC_EXPECTED)):
    try:
        hydrate_dataset(LOCAL_RAW_DIR, DATA_DIR, name, expected)
    except RuntimeError as exc:
        print("HYDRATE FAILED:", exc)
        raise

result = subprocess.run([
    sys.executable,
    str(REPO_DIR / "scripts" / "organize_datasets.py"),
    "--raw-dir", str(LOCAL_RAW_DIR),
    "--data-dir", str(LOCAL_DATA_DIR),
], cwd=str(REPO_DIR))
assert result.returncode == 0, "organize_datasets.py failed"
print("splits ready at", LOCAL_DATA_DIR)

## Step 4 - Measure the real-world gap (the headline number)

Score the Sprint 1 baseline (trained on clean lab photos) on the **PlantDoc test set** - real field
photos. `--map-to-pv` translates PlantDoc classes into PlantVillage labels so the 38-class model can
be scored. This appends the `plantdoc_test` row for `baseline_pv_only_no_aug` and writes
`cm_baseline_pv_only_no_aug.png`. **Expect a big drop vs the 0.96/0.95 PlantVillage numbers - that
drop IS the thesis.**

In [ ]:
cmd = [
    sys.executable, "-m", "ml.evaluate",
    "--checkpoint", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--split", "test",
    "--map-to-pv",
    "--results", str(RESULTS_DIR / "ablation_results.csv"),
    "--variant", "baseline_pv_only_no_aug",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.evaluate failed"

## Step 5 - Stage 2 fine-tune, variant 3 (`pv_plus_plantdoc`)

Warm-start from `best_plantvillage_stage1.pt`, unfreeze the last 2 parameter-bearing backbone blocks
(`--unfreeze-blocks 2`), fine-tune on PlantDoc train (mapped to PV labels) at a low LR:
`--lr 1e-4` for the backbone, `--head-lr 1e-3` for the head. No augmentation. Artifacts:
`plantdoc_stage2_epochNN.pt` + `best_plantdoc_stage2.pt`. If the GPU session is tight, cut
`--epochs` to 5 - the comparison still works.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--unfreeze-blocks", "2",
    "--lr", "1e-4",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--tag", "stage2",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("best fine-tuned checkpoint:", CHECKPOINT_DIR / "best_plantdoc_stage2.pt")

## Step 6 - Evaluate variant 3 on BOTH test sets

Two rows: `pv_plus_plantdoc` on `plantdoc_test` (did the gap shrink?) and on `plantvillage_test`
(did it forget the lab data?). Both appended to the CSV, each gets its own confusion matrix PNG.

In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantdoc_stage2.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", "pv_plus_plantdoc",
    ] + extra
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    assert result.returncode == 0, f"ml.evaluate failed for {dataset}"

## Step 7 - Stage 2 fine-tune, variant 4 (`both`)

Same as variant 3 **plus** augmentation (`--augment`). Artifacts:
`plantdoc_stage2_aug_epochNN.pt` + `best_plantdoc_stage2_aug.pt`.

In [ ]:
cmd = [
    sys.executable, "-m", "ml.train",
    "--data-dir", str(LOCAL_DATA_DIR),
    "--dataset", "plantdoc",
    "--map-to-pv",
    "--init-from", str(CHECKPOINT_DIR / "best_plantvillage_stage1.pt"),
    "--unfreeze-blocks", "2",
    "--lr", "1e-4",
    "--head-lr", "1e-3",
    "--epochs", "10",
    "--batch-size", "32",
    "--checkpoint-dir", str(CHECKPOINT_DIR),
    "--num-workers", "2",
    "--augment",
    "--tag", "stage2_aug",
]
result = subprocess.run(cmd, cwd=str(REPO_DIR))
assert result.returncode == 0, "ml.train failed"
print("best fine-tuned checkpoint:", CHECKPOINT_DIR / "best_plantdoc_stage2_aug.pt")

## Step 8 - Evaluate variant 4 on BOTH test sets

Same as Step 6, checkpoint `best_plantdoc_stage2_aug.pt`, variant `both`.

In [ ]:
for dataset, extra in [("plantdoc", ["--map-to-pv"]), ("plantvillage", [])]:
    cmd = [
        sys.executable, "-m", "ml.evaluate",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantdoc_stage2_aug.pt"),
        "--data-dir", str(LOCAL_DATA_DIR),
        "--dataset", dataset,
        "--split", "test",
        "--results", str(RESULTS_DIR / "ablation_results.csv"),
        "--variant", "both",
    ] + extra
    result = subprocess.run(cmd, cwd=str(REPO_DIR))
    assert result.returncode == 0, f"ml.evaluate failed for {dataset}"

## Step 9 - The verdict: did fine-tuning close the gap?

All rows are in `ablation_results.csv`. Key comparisons (all on the **same** 5,459-image PV test / mapped PlantDoc test):

- baseline on `plantvillage_test` vs baseline on `plantdoc_test` = **the gap**.
- `pv_plus_plantdoc` and `both` on `plantdoc_test` = did fine-tuning help field photos?
- `pv_plus_plantdoc`/`both` on `plantvillage_test` = did we forget the lab data?

In [ ]:
import pandas as pd

df = pd.read_csv(str(RESULTS_DIR / "ablation_results.csv")).drop_duplicates()
cols = ["variant", "dataset", "accuracy", "precision", "recall", "f1"]
print(df[cols].to_string(index=False))

key = lambda v, d: df[(df["variant"] == v) & (df["dataset"] == d)]
before_pv = key("baseline_pv_only_no_aug", "plantvillage_test")
before_pd = key("baseline_pv_only_no_aug", "plantdoc_test")
after_v3 = key("pv_plus_plantdoc", "plantdoc_test")
after_v4 = key("both", "plantdoc_test")

print("\nReal-world gap (PlantDoc field photos):")
if not before_pd.empty:
    g = before_pd.iloc[0]
    print(f"  baseline on PlantDoc test:   accuracy {g['accuracy']:.4f} | f1 {g['f1']:.4f}")
if not before_pv.empty:
    b = before_pv.iloc[0]
    print(f"  baseline on PlantVillage:    accuracy {b['accuracy']:.4f} | f1 {b['f1']:.4f}")
    if not before_pd.empty:
        print(f"  field gap before fine-tune:  {g['f1'] - b['f1']:+.4f} f1")
for label, row in [("  pv_plus_plantdoc on PlantDoc", after_v3), ("  both on PlantDoc", after_v4)]:
    if not row.empty:
        r = row.iloc[0]
        print(f"{label}: accuracy {r['accuracy']:.4f} | f1 {r['f1']:.4f}")

improved = [r.iloc[0]['f1'] for r in (after_v3, after_v4) if not r.empty and r.iloc[0]['f1'] > before_pd.iloc[0]['f1']]
if not before_pd.empty and improved:
    print(f"\nSprint 4 DONE - fine-tuning raised PlantDoc field F1 above baseline by up to "
          f"{max(improved) - before_pd.iloc[0]['f1']:+.4f}")
else:
    print("\nSprint 4 NOT met yet - fine-tuning did not beat the baseline on PlantDoc test")

## Step 10 - Predict a field photo (done-when)

Classify a couple of real PlantDoc test images with the fine-tuned model. Labels print in
PlantVillage class space (the mapped label) - a correct label means the fine-tune mapped the field
photo onto the right disease.

In [ ]:
from pathlib import Path

images = sorted((LOCAL_DATA_DIR / "plantdoc" / "test").glob("*/*.jpg"))[:3]
for image in images:
    cmd = [
        sys.executable, "-m", "ml.predict",
        "--checkpoint", str(CHECKPOINT_DIR / "best_plantdoc_stage2_aug.pt"),
        "--image", str(image),
        "--topk", "3",
    ]
    subprocess.run(cmd, cwd=str(REPO_DIR))
    print("  (true class folder:", image.parent.name, ")")

## Where things live

**On Google Drive (durable):**
```
folium/checkpoints/best_plantdoc_stage2.pt          variant 3 (pv_plus_plantdoc)
folium/checkpoints/best_plantdoc_stage2_aug.pt      variant 4 (both)
folium/results/ablation_results.csv                 all rows (the paper's source of truth)
folium/results/cm_baseline_pv_only_no_aug.png       baseline on PlantDoc (the gap)
folium/results/cm_pv_plus_plantdoc.png              variant 3 on PlantDoc
folium/results/cm_both.png                          variant 4 on PlantDoc
```
Every number in the CSV comes from an actual logged run - nothing fabricated.